# Dataset construction for renewal-risk modelling

This notebook builds the modelling dataset used by the classification, severity and pricing notebooks that follow (prefixed 02 to 05). It combines advertiser data, transactional data, fixed fees and publisher engagement data into a single advertiser-group level dataset, together with a rolling monthly observation panel.

The unit of analysis is fixed throughout at advertiser group by effective business unit as this is how the contracts are negotiated - clients across multiple business units are managed as "Global" business units defined later in the dataset creation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

BASE_PATH   = Path(r"C:\Users\ian.forbes\FinalProject")
OUTPUT_PATH = BASE_PATH / "processed"
OUTPUT_PATH.mkdir(exist_ok=True)

GROUP_KEYS = ['AdvertiserGroup', 'EffectiveBusinessUnit']

## 2. Source transactions

Transaction extracts are supplied as one CSV per calendar year. The files are read in a loop and stacked into a single frame before any further processing.

In [2]:
TRANSACTION_COLUMNS = [
    'AdvertiserKey', 'AdvertiserUID', 'BusinessUnitId',
    'TransactionYearMonth', 'TransactionYear', 'TransactionMonth',
    'CommissionYearMonth', 'CommissionStatus', 'TransactionType',
    'Transactions', 'SalesAmountEUR', 'CommissionEUR', 'NetworkFeeEUR',
    'RecognisedNetworkFeeEUR', 'MonthlyFeeEUR',
]

trans_frames = []
for year in [2023, 2024, 2025, 2026]:
    df = pd.read_csv(BASE_PATH / "Data" / f"{year}TransactionsTable.csv", usecols=TRANSACTION_COLUMNS)
    trans_frames.append(df)
    print(f"  {year}: {len(df):,} rows")

transactions = pd.concat(trans_frames, ignore_index=True) #ingnore_index=True to reset the index after concatenation
del trans_frames
print(f"Total rows: {len(transactions):,}")

advertiser = pd.read_csv(BASE_PATH / "Data" / "Advertiser.csv", low_memory=False) #
fixed_fees  = pd.read_csv(BASE_PATH / "Data" / "FixedFees.csv")
print(f"Advertiser: {len(advertiser):,} rows | FixedFees: {len(fixed_fees):,} rows")

  2023: 855,082 rows
  2024: 880,818 rows
  2025: 869,113 rows
  2026: 295,707 rows
Total rows: 2,900,720
Advertiser: 42,531 rows | FixedFees: 560,469 rows


## 3. Advertiser reference table

The advertiser table is cleaned and produces the elements needed for future analysis:

- Only AW-sourced rows. These are relevant to the business in question and excludes affiliated companies.
- Globally managed programmes are rolled up to a single `Global` business unit rather than being left under whichever country business unit they happen to be tagged with, so that the grain reflects how the account is actually managed.
- Platform tier is converted to a numeric rank (Legacy = 0, Access = 1, Accelerate = 2, Advanced = 3) so it can be used as a model feature. Null values are treated as Legacy, as these are pre-MarTech-migration clients.
- Service tier is ranked on the same principle, but the `Global` prefix is stripped first because it describes scope rather than service level. Where a client carries compound, semicolon-separated values, the highest ranking value is taken.
- Agency-managed clients are flagged separately: they use the technology, but a third-party agency runs the account rather than our own service team.
- A populated migration date indicates a programme that moved across from ShareASale, which is retained as a flag.
- Exclusivity is retained as a feature on the expectation that a structural commitment to the network is associated with lower decline risk than a non-exclusive arrangement.

In [3]:
advertiser = advertiser[advertiser['AdvertiserSourceID'] == 'AW'].copy()

advertiser['LaunchDate']    = pd.to_datetime(advertiser['LaunchDate'],    errors='coerce') #coerce will convert invalids to NaT
advertiser['CloseDate']     = pd.to_datetime(advertiser['CloseDate'],     errors='coerce')
advertiser['MigrationDate'] = pd.to_datetime(advertiser['MigrationDate'], errors='coerce')

advertiser['EffectiveBusinessUnit'] = advertiser.apply(
    lambda r: 'Global' if r['GloballyManaged'] == 'Globally Managed' else r['BusinessUnit'],
    axis=1
)

advertiser['platform_rank'] = advertiser['PlatformOffers'].map(
    {'Access': 1, 'Accelerate': 2, 'Advanced': 3}
).fillna(0)

advertiser['is_global_service_label'] = (
    advertiser['ManagementServices'].str.contains('Global', na=False).astype(int)
)
advertiser['ManagementServices_clean'] = (
    advertiser['ManagementServices'].str.replace('Global ', '', regex=False)
)

_svc_rank_map = {'No Service': 0, 'Essentials': 1, 'Core': 2, 'Premium': 3, 'Bespoke': 4}

def get_max_service_rank(value):
    if pd.isna(value):
        return 0
    ranks = [_svc_rank_map.get(c.strip()) for c in value.split(';')]
    ranks = [r for r in ranks if r is not None]
    return max(ranks) if ranks else None

advertiser['service_rank'] = (
    advertiser['ManagementServices_clean'].apply(get_max_service_rank).fillna(1)
)

advertiser['is_agency_managed'] = advertiser['Agency'].notna().astype(int)

def categorise_service_relationship(row):
    if row['service_rank'] > 0:   return 'Awin Managed'
    if row['is_agency_managed']:  return 'Agency Managed'
    return 'Self Service'

advertiser['service_relationship_type'] = advertiser.apply(
    categorise_service_relationship, axis=1
)

advertiser['former_sas_flag'] = advertiser['MigrationDate'].notna().astype(int)

advertiser['is_exclusive'] = (advertiser['Exclusive'] == 'Exclusive').astype(int)

print(advertiser['service_relationship_type'].value_counts())
print(advertiser['Status'].value_counts())
print(f"Exclusive: {advertiser['is_exclusive'].sum():,} of {len(advertiser):,}")

service_relationship_type
Self Service      19737
Awin Managed      10258
Agency Managed     2763
Name: count, dtype: int64
Status
Live              20587
Closed            11287
Hidden              622
OK for network      262
Name: count, dtype: int64
Exclusive: 8,860 of 32,758


## 4. Transaction filtering and recognition-rate adjustment

Transactions can be either Confirmed, Pending or Declined - Pending transactions will become either confirmed or declined over time.

- `MonthlyFeeEUR` is dropped here because fixed fee income is taken from the dedicated fixed fees dataset instead - the data in the transaction table is not split by the type of fixed fee hence why it is not used for the analysis.
- Pending rows are scaled by `RecognitionRate`, which is a live snapshot of how much of that pending commission is actually expected to confirm.
- Pending rows with no recognition rate are dropped. These are typically legacy ShareASale clients that have not yet been migrated, so no reliable expected value can be assigned.
- CPI refers to the Conversion Protection Initiative, It is a mechanism for compensating publishers where activity is likely to be under-attributed because of tracking issues.

In [4]:
transactions['TransactionYearMonth'] = pd.to_datetime(transactions['TransactionYearMonth'])

transactions = transactions[
    transactions['CommissionStatus'].isin(['Confirmed', 'Pending'])
].copy()
print(f"After CommissionStatus filter: {len(transactions):,} rows")

transactions = transactions.drop(columns=['MonthlyFeeEUR'])

recognition_lookup = advertiser[['AdvertiserUID', 'RecognitionRate']].drop_duplicates()
transactions = transactions.merge(recognition_lookup, on='AdvertiserUID', how='left')

pending_mask = transactions['CommissionStatus'] == 'Pending'
drop_mask    = pending_mask & transactions['RecognitionRate'].isnull()
transactions = transactions[~drop_mask].copy()
print(f"After dropping unmatched Pending rows: {len(transactions):,}")

transactions['CommissionEUR_Adj']  = transactions['CommissionEUR']
transactions['NetworkFeeEUR_Adj']  = transactions['NetworkFeeEUR']
transactions.loc[pending_mask, 'CommissionEUR_Adj'] = (
    transactions.loc[pending_mask, 'CommissionEUR']
    * transactions.loc[pending_mask, 'RecognitionRate']
)
transactions.loc[pending_mask, 'NetworkFeeEUR_Adj'] = (
    transactions.loc[pending_mask, 'NetworkFeeEUR']
    * transactions.loc[pending_mask, 'RecognitionRate']
)

transactions['is_cpi'] = (transactions['TransactionType'] == 'Probabilistic').astype(int)

cpi_summary = transactions.groupby('AdvertiserUID').apply(
    lambda g: pd.Series({
        'cpi_commission_total': g.loc[g['is_cpi'] == 1, 'CommissionEUR_Adj'].sum(),
        'total_commission': g['CommissionEUR_Adj'].sum(),
    })
).reset_index()
cpi_summary['cpi_share_of_commission'] = (
    cpi_summary['cpi_commission_total'] / cpi_summary['total_commission']
).fillna(0)
cpi_summary['has_cpi_flag'] = (cpi_summary['cpi_share_of_commission'] > 0).astype(int)

advertiser = advertiser.merge(
    cpi_summary[['AdvertiserUID', 'cpi_share_of_commission', 'has_cpi_flag']],
    on='AdvertiserUID', how='left'
)
advertiser['cpi_share_of_commission'] = advertiser['cpi_share_of_commission'].fillna(0)
advertiser['has_cpi_flag']            = advertiser['has_cpi_flag'].fillna(0).astype(int)

After CommissionStatus filter: 2,127,048 rows
After dropping unmatched Pending rows: 2,121,252


## 5. Commission by service relationship

A summary check of how commission is distributed across service relationship types, used to confirm the population looks as expected before aggregation.

In [5]:
service_lookup    = advertiser[['AdvertiserUID', 'service_relationship_type']].drop_duplicates()
trans_with_svc    = transactions.merge(service_lookup, on='AdvertiserUID', how='left')
commission_by_svc = trans_with_svc.groupby('service_relationship_type').agg(
    advertiser_count=('AdvertiserUID', 'nunique'),
    total_commission=('CommissionEUR_Adj', 'sum')
)
commission_by_svc['pct_advertisers'] = (
    commission_by_svc['advertiser_count'] / commission_by_svc['advertiser_count'].sum() * 100
).round(1)
commission_by_svc['pct_commission'] = (
    commission_by_svc['total_commission'] / commission_by_svc['total_commission'].sum() * 100
).round(1)
print(commission_by_svc)
del trans_with_svc

                           advertiser_count  total_commission  pct_advertisers  pct_commission
service_relationship_type                                                                     
Agency Managed                         2693      1.582493e+08              8.4             3.7
Awin Managed                          10135      3.772069e+09             31.7            87.7
Self Service                          19129      3.708077e+08             59.9             8.6


## 6. Group lookup and point-in-time programme status

Two helper functions establish which programmes count as live at a given observation month, so that features can be constructed on a point-in-time basis rather than using present-day status.

- A programme counts as live at an observation month if its status is Live or Hidden, or if it is Closed with a close date falling on or after that month. Hidden programmes were checked and found to be genuinely active: approximately 95 per cent have transactions, representing around EUR 518m of commission, so they belong in the live population rather than being excluded.
- The `OK for network` status is left out. It is a pre-launch holding state accounting for under 0.1 per cent of commission.
- Partially closed groups are those where one programme has closed within the previous 12 months while the group still has other programmes live. This is a feature because it is an early indicator of a group already contracting.

In [6]:
def get_live_programmes_at(advertiser_df, observation_month):
    mask = (
        advertiser_df['Status'].isin(['Live', 'Hidden'])
        | (
            (advertiser_df['Status'] == 'Closed')
            & advertiser_df['CloseDate'].notna()
            & (advertiser_df['CloseDate'] >= observation_month)
        )
    )
    return set(advertiser_df.loc[mask, 'AdvertiserUID'])

def get_partially_closed_groups(advertiser_df, group_keys, observation_month):
    window_start = observation_month - pd.DateOffset(months=12)

    closed_in_window = advertiser_df[
        (advertiser_df['Status'] == 'Closed')
        & advertiser_df['CloseDate'].notna()
        & (advertiser_df['CloseDate'] >= window_start)
        & (advertiser_df['CloseDate'] < observation_month)
    ]
    live_uids  = get_live_programmes_at(advertiser_df, observation_month)
    live_now   = advertiser_df[advertiser_df['AdvertiserUID'].isin(live_uids)]

    closed_set = set(map(tuple, closed_in_window[group_keys].values))
    live_set   = set(map(tuple, live_now[group_keys].values))
    return closed_set & live_set

## 7. Monthly aggregation: transactions

Transactions are aggregated to advertiser group, business unit and month.

Transaction dates are end-of-month while fixed fee dates are first-of-month. Both are converted to a consistent monthly period here so that the later merge aligns correctly.

In [7]:
transactions_grouped = transactions.merge(
    advertiser[['AdvertiserUID'] + GROUP_KEYS],
    on='AdvertiserUID',
    how='inner'
)
print(f"Transactions after group merge: {len(transactions_grouped):,}")

transactions_grouped['cpi_commission_row'] = (
    transactions_grouped['CommissionEUR_Adj'] * transactions_grouped['is_cpi']
)

transactions_grouped['MonthKey'] = (
    transactions_grouped['TransactionYearMonth'].dt.to_period('M')
)

monthly_transactions = transactions_grouped.groupby(
    GROUP_KEYS + ['MonthKey']
).agg(
    CommissionEUR_Adj=('CommissionEUR_Adj', 'sum'),
    NetworkFeeEUR_Adj=('NetworkFeeEUR_Adj', 'sum'),
    RecognisedNetworkFeeEUR=('RecognisedNetworkFeeEUR', 'sum'),
    SalesAmountEUR=('SalesAmountEUR', 'sum'),
    TransactionCount=('Transactions', 'sum'),
    CPICommissionEUR=('cpi_commission_row', 'sum'),
).reset_index()

monthly_transactions['cpi_share_of_commission_month'] = (
    monthly_transactions['CPICommissionEUR'] / monthly_transactions['CommissionEUR_Adj']
).fillna(0)

print(f"Monthly transactions: {len(monthly_transactions):,} rows")

Transactions after group merge: 1,859,122
Monthly transactions: 646,759 rows


## 8. Monthly aggregation: fixed fees

Fixed fees are aggregated to the same monthly grain as transactions.

Advertiser and advertiser group columns had to be dropped from the fixed fees frame before the merge, to avoid clashing with the identically named columns already carried on the advertiser table.

In [8]:
fixed_fees['MetricMonth'] = pd.to_datetime(
    fixed_fees['MetricMonth'], format='%d/%m/%Y', errors='coerce'
)

fixed_fees_grouped = fixed_fees.drop(
    columns=['Advertiser', 'AdvertiserGroup'], errors='ignore' #ignore errors if columns don't exist
).merge(advertiser[['AdvertiserUID'] + GROUP_KEYS], on='AdvertiserUID', how='inner')

fixed_fees_grouped['MonthKey'] = fixed_fees_grouped['MetricMonth'].dt.to_period('M')

monthly_fixed_fees = fixed_fees_grouped.groupby(
    GROUP_KEYS + ['MonthKey']
).agg(
    PlatformGPEUR=('PlatformGPEUR', 'sum'),
    ServiceGPEUR=('ServiceGPEUR', 'sum'),
    CTFGPEUR=('CTFGPEUR', 'sum'),
    TenancyGPEUR=('TenancyGPEUR', 'sum'),
    LegacyFeesGPEUR=('LegacyFeesGPEUR', 'sum'),
    SASGPEUR=('SASGPEUR', 'sum'),
    SetupGPEUR=('SetupGPEUR', 'sum'),
    OtherGPEUR=('OtherGPEUR', 'sum'),
    UnmappedGPEUR=('UnmappedGPEUR', 'sum'),
    FixedGPEUR=('FixedGPEUR', 'sum'),
).reset_index()

print(f"Monthly fixed fees: {len(monthly_fixed_fees):,} rows")

Monthly fixed fees: 520,706 rows


## 9. Combined monthly table, Core GP and effective take rate

Transactions and fixed fees are joined into a single monthly table, and the core commercial measures used throughout the project are derived.

- An outer join is used because a group may have fees in a month with no transactions, or more likely transactions with no fixed fees. 
- Core GP is defined as tracking plus platform plus service plus CTF. Tenancy is excluded because it is a placement fee rather than ongoing monetisation, and SAS income is excluded as legacy ShareASale revenue and kept separate.
- The effective take rate (ETR) is Core GP as a share of commission. This is a key pricing signal, expressing how much of a client's activity is actually monetised, and it allows clients on fixed fees to be compared like-for-like with clients who have none.
- Fixed fee share of GP is retained because the higher it is, the more of a client's GP is locked in regardless of volume, and therefore the less exposed that GP is if commission falls.

In [9]:
monthly_panel_base = monthly_transactions.merge(
    monthly_fixed_fees, on=GROUP_KEYS + ['MonthKey'], how='outer'
)

monthly_panel_base['TransactionYearMonth'] = (
    monthly_panel_base['MonthKey'].dt.to_timestamp('M')
)
monthly_panel_base = monthly_panel_base.drop(columns=['MonthKey'])

fill_zero_cols = [
    'CommissionEUR_Adj', 'NetworkFeeEUR_Adj', 'RecognisedNetworkFeeEUR',
    'SalesAmountEUR', 'TransactionCount', 'CPICommissionEUR',
    'cpi_share_of_commission_month', 'PlatformGPEUR', 'ServiceGPEUR',
    'CTFGPEUR', 'TenancyGPEUR', 'LegacyFeesGPEUR', 'SASGPEUR',
    'SetupGPEUR', 'OtherGPEUR', 'UnmappedGPEUR', 'FixedGPEUR',
]
monthly_panel_base[fill_zero_cols] = monthly_panel_base[fill_zero_cols].fillna(0)

monthly_panel_base['CoreGPEUR'] = (
    monthly_panel_base['RecognisedNetworkFeeEUR']
    + monthly_panel_base['PlatformGPEUR']
    + monthly_panel_base['ServiceGPEUR']
    + monthly_panel_base['CTFGPEUR']
)

monthly_panel_base['TotalFixedGPEUR'] = (
    monthly_panel_base['PlatformGPEUR']
    + monthly_panel_base['ServiceGPEUR']
    + monthly_panel_base['CTFGPEUR']
)

monthly_panel_base['TotalGPIncludingSAS'] = (
    monthly_panel_base['CoreGPEUR'] + monthly_panel_base['SASGPEUR']
)

monthly_panel_base['ETR'] = (
    monthly_panel_base['CoreGPEUR'] / monthly_panel_base['CommissionEUR_Adj']
).replace([float('inf'), -float('inf')], None)

monthly_panel_base['FixedFeeShareOfGP'] = (
    monthly_panel_base['TotalFixedGPEUR'] / monthly_panel_base['CoreGPEUR']
).replace([float('inf'), -float('inf')], None)

print(f"Monthly base table: {len(monthly_panel_base):,} rows, "
      f"{monthly_panel_base.groupby(GROUP_KEYS).ngroups:,} group/BU combinations")
print(f"Date range: {monthly_panel_base['TransactionYearMonth'].min()} to "
      f"{monthly_panel_base['TransactionYearMonth'].max()}")

Monthly base table: 663,443 rows, 28,472 group/BU combinations
Date range: 2023-01-31 00:00:00 to 2026-06-30 00:00:00


## 10. Outputs of the data created so far

The intermediate tables are written to disk so that we do not need to rebuild again

In [10]:
monthly_panel_base.to_csv(OUTPUT_PATH / "monthly_panel_base.csv", index=False)
advertiser.to_csv(OUTPUT_PATH / "advertiser_cleaned.csv", index=False)

## 11. Rolling observation panel

A rolling panel is built across multiple observation months rather than a single snapshot, so that the same feature definitions can be reused for other observation points in future work.

All momentum features are constructed year-on-year to control for seasonality, across three windows:

- **L12M** is the full year against the year before it, giving long-term direction.
- **L6M** is the most recent six months against the same six months a year earlier, giving mid-term direction.
- **L3M** is the most recent quarter against the same quarter a year earlier, giving recent direction.

The trailing 12 months are additionally broken into four quarterly year-on-year momentum measures (Q1 to Q4, counting back from the feature end date), so that the later modelling can examine the trajectory quarter by quarter rather than only recent-against-full-year.

- Only programmes actually live at the observation month are included, using the point-in-time logic defined earlier.
- Year-on-year features are only constructed where a full prior year of data exists to compare against; where it does not, the momentum features are left as null rather than imputed at this stage.
- The 12-month and 6-month momentum measures are retained in the panel for reference but are not used by the model, because both are nested with three-month momentum (r = 0.98 and r = 0.996 respectively). Three-month momentum is the measure carried into modelling, being the most sensitive to recent deterioration while remaining a fair seasonal comparison.
- Q1 of the quarterly breakdown is identical to three-month momentum; it is given its own column name so that the classification notebook can reference `momentum_q1_yoy` directly.
- Consecutive declining months and the count of negative-growth months are calculated month-on-month rather than year-on-year. This is appropriate here because these features measure the consistency of direction, not the size of the movement.
- The target is defined as a decline where forward 12-month commission falls more than the specified threshold below trailing 12-month commission.
- Where a programme closed during the target window, the target label is set to null rather than the group being dropped. A closure is a structural churn event rather than the performance decline within an ongoing relationship that this model is intended to detect. Retaining the row means that where the group has other programmes still live, those continue to contribute normally.

In [11]:
OBS_START              = pd.Timestamp('2024-01-01')
OBS_END                = pd.Timestamp('2025-06-01')
DECLINE_THRESHOLD      = 0.10
SENSITIVITY_THRESHOLDS = [0.05, 0.10, 0.15]

def end_of_month(ts):
    return ts + pd.offsets.MonthEnd(0)

obs_months = pd.date_range(start=OBS_START, end=OBS_END, freq='MS') #freq  MS is month start
panel_rows = []

for obs_month in obs_months:

    feature_end      = obs_month - relativedelta(months=1)
    feature_start    = obs_month - relativedelta(months=12)
    prior_yr_start   = feature_start - relativedelta(months=12)
    has_prior_year   = (prior_yr_start >= pd.Timestamp('2023-01-01'))

    feature_end_eom   = end_of_month(feature_end)
    feature_start_eom = end_of_month(feature_start)
    target_start_eom  = end_of_month(obs_month)
    target_end_eom    = end_of_month(obs_month + relativedelta(months=11))
    prior_end_eom     = end_of_month(feature_start - relativedelta(months=1))
    prior_start_eom   = end_of_month(prior_yr_start)

    live_uids = get_live_programmes_at(advertiser, obs_month)
    live_groups = advertiser[
        advertiser['AdvertiserUID'].isin(live_uids)
    ][GROUP_KEYS].drop_duplicates()

    if len(live_groups) == 0:
        continue

    live_idx   = set(map(tuple, live_groups.values))
    panel_live = monthly_panel_base[
        monthly_panel_base.set_index(GROUP_KEYS).index.isin(live_idx)
    ].copy()

    feature_data = panel_live[
        (panel_live['TransactionYearMonth'] >= feature_start_eom) &
        (panel_live['TransactionYearMonth'] <= feature_end_eom)
    ]
    target_data = panel_live[
        (panel_live['TransactionYearMonth'] >= target_start_eom) &
        (panel_live['TransactionYearMonth'] <= target_end_eom)
    ]
    prior_data = panel_live[
        (panel_live['TransactionYearMonth'] >= prior_start_eom) &
        (panel_live['TransactionYearMonth'] <= prior_end_eom)
    ] if has_prior_year else None

    feature_agg = feature_data.groupby(GROUP_KEYS).agg(
        commission_l12m=('CommissionEUR_Adj', 'sum'),
        sales_l12m=('SalesAmountEUR', 'sum'),
        tracking_gp_l12m=('RecognisedNetworkFeeEUR', 'sum'),
        platform_gp_l12m=('PlatformGPEUR', 'sum'),
        service_gp_l12m=('ServiceGPEUR', 'sum'),
        ctf_gp_l12m=('CTFGPEUR', 'sum'),
        tenancy_gp_l12m=('TenancyGPEUR', 'sum'),
        sas_gp_l12m=('SASGPEUR', 'sum'),
        core_gp_l12m=('CoreGPEUR', 'sum'),
        total_gp_including_sas_l12m=('TotalGPIncludingSAS', 'sum'),
        total_fixed_gp_l12m=('TotalFixedGPEUR', 'sum'),
        cpi_commission_l12m=('CPICommissionEUR', 'sum'),
        transaction_count_l12m=('TransactionCount', 'sum'),
        commission_months_count=('CommissionEUR_Adj', 'count'),
        commission_std=('CommissionEUR_Adj', 'std'),
        commission_max_month=('CommissionEUR_Adj', 'max'),
        zero_commission_months=('CommissionEUR_Adj', lambda x: (x == 0).sum()),
        etr_mean_l12m=('ETR', 'mean'),
        fixed_fee_share_mean_l12m=('FixedFeeShareOfGP', 'mean'),
    ).reset_index()

    feature_agg['active_months_l12m'] = 12 - feature_agg['zero_commission_months']
    feature_agg['cpi_share_l12m'] = (
        feature_agg['cpi_commission_l12m'] / feature_agg['commission_l12m']
    ).fillna(0)
    feature_agg['commission_cv_l12m'] = (
        feature_agg['commission_std'] / (feature_agg['commission_l12m'] / 12)
    ).replace([float('inf'), -float('inf')], None)
    feature_agg['commission_peak_concentration_l12m'] = (
        feature_agg['commission_max_month'] / feature_agg['commission_l12m']
    ).replace([float('inf'), -float('inf')], None)
    feature_agg['commission_avg_monthly_l12m'] = feature_agg['commission_l12m'] / 12

    if prior_data is not None and len(prior_data) > 0:

        prior_agg = prior_data.groupby(GROUP_KEYS).agg(
            commission_prior_year=('CommissionEUR_Adj', 'sum'),
            tracking_gp_prior_year=('RecognisedNetworkFeeEUR', 'sum'),
            core_gp_prior_year=('CoreGPEUR', 'sum'),
            etr_mean_prior_year=('ETR', 'mean'),
        ).reset_index()
        feature_agg = feature_agg.merge(prior_agg, on=GROUP_KEYS, how='left')

        feature_agg['commission_yoy_growth'] = (
            (feature_agg['commission_l12m'] - feature_agg['commission_prior_year'])
            / feature_agg['commission_prior_year'].abs()
        ).replace([float('inf'), -float('inf')], None)

        feature_agg['tracking_gp_yoy_growth'] = (
            (feature_agg['tracking_gp_l12m'] - feature_agg['tracking_gp_prior_year'])
            / feature_agg['tracking_gp_prior_year'].abs()
        ).replace([float('inf'), -float('inf')], None)

        feature_agg['etr_yoy_change'] = (
            feature_agg['etr_mean_l12m'] - feature_agg['etr_mean_prior_year']
        )

        recent_6m_start = end_of_month(feature_end - relativedelta(months=5))
        prior_6m_start  = end_of_month(feature_end - relativedelta(months=17))
        prior_6m_end    = end_of_month(feature_end - relativedelta(months=12))

        recent_6m = feature_data[
            feature_data['TransactionYearMonth'] >= recent_6m_start
        ].groupby(GROUP_KEYS)['CommissionEUR_Adj'].sum().rename('commission_recent_6m')

        prior_6m = prior_data[
            (prior_data['TransactionYearMonth'] >= prior_6m_start) &
            (prior_data['TransactionYearMonth'] <= prior_6m_end)
        ].groupby(GROUP_KEYS)['CommissionEUR_Adj'].sum().rename('commission_prior_6m_yoy')

        mom6 = pd.concat([recent_6m, prior_6m], axis=1).fillna(0)
        mom6['momentum_6m_yoy'] = (
            (mom6['commission_recent_6m'] - mom6['commission_prior_6m_yoy'])
            / mom6['commission_prior_6m_yoy'].abs()
        ).replace([float('inf'), -float('inf')], None)
        feature_agg = feature_agg.merge(
            mom6[['momentum_6m_yoy', 'commission_recent_6m',
                  'commission_prior_6m_yoy']].reset_index(),
            on=GROUP_KEYS, how='left'
        )

        recent_3m_start = end_of_month(feature_end - relativedelta(months=2))
        prior_3m_start  = end_of_month(feature_end - relativedelta(months=14))
        prior_3m_end    = end_of_month(feature_end - relativedelta(months=12))

        recent_3m = feature_data[
            feature_data['TransactionYearMonth'] >= recent_3m_start
        ].groupby(GROUP_KEYS)['CommissionEUR_Adj'].sum().rename('commission_recent_3m')

        prior_3m = prior_data[
            (prior_data['TransactionYearMonth'] >= prior_3m_start) &
            (prior_data['TransactionYearMonth'] <= prior_3m_end)
        ].groupby(GROUP_KEYS)['CommissionEUR_Adj'].sum().rename('commission_prior_3m_yoy')

        mom3 = pd.concat([recent_3m, prior_3m], axis=1).fillna(0)
        mom3['momentum_3m_yoy'] = (
            (mom3['commission_recent_3m'] - mom3['commission_prior_3m_yoy'])
            / mom3['commission_prior_3m_yoy'].abs()
        ).replace([float('inf'), -float('inf')], None)
        feature_agg = feature_agg.merge(
            mom3[['momentum_3m_yoy', 'commission_recent_3m',
                  'commission_prior_3m_yoy']].reset_index(),
            on=GROUP_KEYS, how='left'
        )

        feature_agg['momentum_q1_yoy'] = feature_agg['momentum_3m_yoy']

        for q, offset in [(2, 3), (3, 6), (4, 9)]:
            recent_start = end_of_month(feature_end - relativedelta(months=offset + 2))
            recent_end   = end_of_month(feature_end - relativedelta(months=offset))
            prior_start  = end_of_month(feature_end - relativedelta(months=offset + 14))
            prior_end    = end_of_month(feature_end - relativedelta(months=offset + 12))

            recent_q = feature_data[
                (feature_data['TransactionYearMonth'] >= recent_start) &
                (feature_data['TransactionYearMonth'] <= recent_end)
            ].groupby(GROUP_KEYS)['CommissionEUR_Adj'].sum().rename(f'commission_recent_q{q}')

            prior_q = prior_data[
                (prior_data['TransactionYearMonth'] >= prior_start) &
                (prior_data['TransactionYearMonth'] <= prior_end)
            ].groupby(GROUP_KEYS)['CommissionEUR_Adj'].sum().rename(f'commission_prior_q{q}_yoy')

            momq = pd.concat([recent_q, prior_q], axis=1).fillna(0)
            momq[f'momentum_q{q}_yoy'] = (
                (momq[f'commission_recent_q{q}'] - momq[f'commission_prior_q{q}_yoy'])
                / momq[f'commission_prior_q{q}_yoy'].abs()
            ).replace([float('inf'), -float('inf')], None)
            feature_agg = feature_agg.merge(
                momq[[f'momentum_q{q}_yoy']].reset_index(),
                on=GROUP_KEYS, how='left'
            )

    else:

        for col in ['commission_prior_year', 'tracking_gp_prior_year',
                    'core_gp_prior_year', 'etr_mean_prior_year',
                    'commission_yoy_growth', 'tracking_gp_yoy_growth',
                    'etr_yoy_change', 'momentum_6m_yoy', 'commission_recent_6m',
                    'commission_prior_6m_yoy', 'momentum_3m_yoy',
                    'commission_recent_3m', 'commission_prior_3m_yoy',
                    'momentum_q1_yoy', 'momentum_q2_yoy',
                    'momentum_q3_yoy', 'momentum_q4_yoy']:
            feature_agg[col] = None

    feature_agg['has_prior_year_features'] = int(has_prior_year)

    monthly_commission = (
        feature_data.sort_values(GROUP_KEYS + ['TransactionYearMonth'])
        .groupby(GROUP_KEYS + ['TransactionYearMonth'])['CommissionEUR_Adj'].sum()
    )

    def count_consecutive_decline(series):
        count = 0
        for v in reversed(series.diff().values[1:]):
            if v < 0: count += 1
            else: break
        return count

    consec = (
        monthly_commission.reset_index()
        .groupby(GROUP_KEYS)['CommissionEUR_Adj']
        .apply(count_consecutive_decline).reset_index()
        .rename(columns={'CommissionEUR_Adj': 'consecutive_declining_months'})
    )
    neg_mths = (
        monthly_commission.reset_index()
        .groupby(GROUP_KEYS)['CommissionEUR_Adj']
        .apply(lambda x: (x.diff() < 0).sum()).reset_index()
        .rename(columns={'CommissionEUR_Adj': 'negative_growth_months_l12m'})
    )
    feature_agg = feature_agg.merge(consec,   on=GROUP_KEYS, how='left')
    feature_agg = feature_agg.merge(neg_mths, on=GROUP_KEYS, how='left')

    target_agg = target_data.groupby(GROUP_KEYS).agg(
        commission_forward_12m=('CommissionEUR_Adj', 'sum')
    ).reset_index()
    feature_agg = feature_agg.merge(target_agg, on=GROUP_KEYS, how='left')

    closed_in_target = advertiser[
        advertiser['CloseDate'].notna() &
        (advertiser['CloseDate'] >= target_start_eom) &
        (advertiser['CloseDate'] <= target_end_eom)
    ][GROUP_KEYS].drop_duplicates()

    closed_in_target_set = set(map(tuple, closed_in_target.values))

    feature_agg['has_closure_in_target'] = feature_agg.apply(
        lambda r: int(
            (r['AdvertiserGroup'], r['EffectiveBusinessUnit'])
            in closed_in_target_set
        ), axis=1
    )
    n_closed = feature_agg['has_closure_in_target'].sum()
    if n_closed > 0:
        print(f"{n_closed:,} groups with closure in within the target period")

    feature_agg['target_decline'] = (
        feature_agg['commission_forward_12m'] <
        feature_agg['commission_l12m'] * (1 - DECLINE_THRESHOLD)
    ).astype(int)

    feature_agg.loc[
        feature_agg['has_closure_in_target'] == 1, 'target_decline'
    ] = np.nan

    for thresh in SENSITIVITY_THRESHOLDS:
        col = f'target_decline_{int(thresh*100)}pct'
        feature_agg[col] = (
            feature_agg['commission_forward_12m'] <
            feature_agg['commission_l12m'] * (1 - thresh)
        ).astype(int)
        feature_agg.loc[
            feature_agg['has_closure_in_target'] == 1, col
        ] = np.nan

    feature_agg['target_decline_pct'] = (
        (feature_agg['commission_l12m'] - feature_agg['commission_forward_12m'])
        / feature_agg['commission_l12m'].abs()
    ).replace([float('inf'), -float('inf')], None)

    feature_agg.loc[
        feature_agg['has_closure_in_target'] == 1, 'target_decline_pct'
    ] = np.nan

    feature_agg['baseline_eligible'] = int(has_prior_year)
    feature_agg['baseline_prediction'] = (
        (feature_agg['commission_yoy_growth'] < 0).astype(int)
        if has_prior_year else None
    )

    prog_count = advertiser[
        advertiser['AdvertiserUID'].isin(live_uids)
    ].groupby(GROUP_KEYS)['AdvertiserUID'].nunique().reset_index()
    prog_count.columns = GROUP_KEYS + ['programme_count']
    feature_agg = feature_agg.merge(prog_count, on=GROUP_KEYS, how='left')

    partially_closed = get_partially_closed_groups(advertiser, GROUP_KEYS, obs_month)
    feature_agg['partially_closed_last_12m'] = feature_agg.apply(
        lambda r: int(
            (r['AdvertiserGroup'], r['EffectiveBusinessUnit']) in partially_closed
        ), axis=1
    )

    live_uids_12m_ago = get_live_programmes_at(
        advertiser, obs_month - relativedelta(months=12)
    )
    prog_12m_ago = advertiser[
        advertiser['AdvertiserUID'].isin(live_uids_12m_ago)
    ].groupby(GROUP_KEYS)['AdvertiserUID'].nunique().reset_index()
    prog_12m_ago.columns = GROUP_KEYS + ['programme_count_12m_ago']
    feature_agg = feature_agg.merge(prog_12m_ago, on=GROUP_KEYS, how='left')
    feature_agg['programme_count_change'] = (
        feature_agg['programme_count'] - feature_agg['programme_count_12m_ago']
    )

    static = advertiser.groupby(GROUP_KEYS).agg(
        sector=('Sector', lambda x: x.iloc[0] if x.nunique() == 1 else 'Multiple'),
        service_rank=('service_rank', 'max'),
        platform_rank=('platform_rank', 'max'),
        service_relationship_type=('service_relationship_type', lambda x: x.iloc[0]),
        is_agency_managed=('is_agency_managed', 'max'),
        former_sas_flag=('former_sas_flag', 'max'),
        has_cpi_flag=('has_cpi_flag', 'max'),
        cpi_share_of_commission=('cpi_share_of_commission', 'max'),
        is_global_service_label=('is_global_service_label', 'max'),
        is_exclusive=('is_exclusive', 'max'),
        launch_date=('LaunchDate', 'min'),
    ).reset_index()

    feature_agg = feature_agg.merge(static, on=GROUP_KEYS, how='left')
    feature_agg['programme_age_months'] = feature_agg['launch_date'].apply(
        lambda d: (obs_month.year - d.year) * 12 + (obs_month.month - d.month)
        if pd.notna(d) else None
    )

    feature_agg['ObservationMonth'] = obs_month
    panel_rows.append(feature_agg)
    print(f"  {obs_month.date()} — {len(feature_agg):,} groups")

1,547 groups with closure in within the target period
  2024-01-01 — 15,475 groups
1,576 groups with closure in within the target period
  2024-02-01 — 15,596 groups
1,601 groups with closure in within the target period
  2024-03-01 — 15,741 groups
1,622 groups with closure in within the target period
  2024-04-01 — 15,929 groups
1,704 groups with closure in within the target period
  2024-05-01 — 16,238 groups
1,766 groups with closure in within the target period
  2024-06-01 — 16,475 groups
1,785 groups with closure in within the target period
  2024-07-01 — 16,726 groups
1,849 groups with closure in within the target period
  2024-08-01 — 16,986 groups
1,927 groups with closure in within the target period
  2024-09-01 — 17,177 groups
2,086 groups with closure in within the target period
  2024-10-01 — 17,371 groups
2,279 groups with closure in within the target period
  2024-11-01 — 17,634 groups
2,439 groups with closure in within the target period
  2024-12-01 — 17,889 groups
2,48

## 12. Panel assembly and target validation

The full panel is assembled and the target variable is checked before modelling.

Row counts are captured before the closure-window rows are dropped, so that the later diagnostic section can report the January 2025 numbers before and after

Rows with a null target are dropped at this point. These are groups with a closure in the target window, where the commission fall will be because of churn and therefore not partial defection.

Excluding shadow-churn clients (clients who decline 100%) was tested but then reverted. The reasoning is that if these clients had been repriced onto more fixed terms while still active, we would have continued collecting the fixed fee even after variable commission fell to zero. That is precisely the value this model is intended to trigger, so removing the clients with the largest potential repricing payoff was the wrong decision for this use case, even though it would be the correct decision for a conventional churn-prediction model.

In [12]:
panel = pd.concat(panel_rows, ignore_index=True)
panel = panel.dropna(subset=['commission_forward_12m'])
panel = panel[panel['programme_count'] > 0]

n_before_closure_jan25 = len(panel[panel['ObservationMonth'] == pd.Timestamp('2025-01-01')])

n_before = len(panel)
panel = panel.dropna(subset=['target_decline'])
n_removed = n_before - len(panel)

print(f"\nClass balance by observation month:")
mb = panel.groupby('ObservationMonth')['target_decline'].agg(total='count', decline='sum')
mb['decline_rate_pct'] = (mb['decline'] / mb['total'] * 100).round(1)
print(mb.to_string())


Class balance by observation month:
                  total  decline  decline_rate_pct
ObservationMonth                                  
2024-01-01        13653   6013.0              44.0
2024-02-01        13730   6103.0              44.5
2024-03-01        13851   6198.0              44.7
2024-04-01        13996   6255.0              44.7
2024-05-01        14216   6284.0              44.2
2024-06-01        14382   6312.0              43.9
2024-07-01        14595   6430.0              44.1
2024-08-01        14816   6463.0              43.6
2024-09-01        14951   6380.0              42.7
2024-10-01        14982   6243.0              41.7
2024-11-01        15053   6111.0              40.6
2024-12-01        15165   6003.0              39.6
2025-01-01        15247   6191.0              40.6
2025-02-01        15255   6240.0              40.9
2025-03-01        15195   6158.0              40.5
2025-04-01        15198   6171.0              40.6
2025-05-01        15283   6249.0             

## 13. Rolling panel output

The completed rolling panel is written to disk for use by the modelling notebooks.

In [13]:
panel.to_csv(OUTPUT_PATH / "observation_panel.csv", index=False)

## 14. Publisher engagement features and final modelling dataset

Publisher engagement data is added and the final single-snapshot modelling dataset is assembled.

Publisher data is exported from a Azure Analysis Services cube as monthly commission by advertiser programme and publisher subvertical, data is not available for 2023, so is covering January 2024 to January 2025. The features derived from it are RFM-adjacent platform engagement signals:

- commission share by funnel stage (lower, mid, upper, technical);
- a funnel diversity score, measuring how many stages a client actually uses; and
- the year-on-year shift in funnel mix, January 2025 against January 2024.

Publisher subverticals are mapped to four funnel stages:

- **Upper** is brand and content publishers
- **Mid** is discovery and comparison
- **Lower** is cashback, discount and loyalty
- **Technical** is infrastructure and direct traffic


In [14]:

print("====PART M - PUBLISHER ENRICHMENT====")


pub_raw = pd.read_csv(BASE_PATH / "Data" / "Publisher.csv", low_memory=False)
print(f"Publisher data loaded: {len(pub_raw):,} rows")
print(f"Columns: {list(pub_raw.columns)}")

pub_raw['AdvertiserUID'] = 'AW' + pub_raw['Advertiser ID'].astype(str).str.strip()

pub_raw = pub_raw.merge(
    advertiser[['AdvertiserUID', 'AdvertiserGroup', 'EffectiveBusinessUnit']
               ].drop_duplicates(subset=['AdvertiserUID']),
    on='AdvertiserUID', how='inner'
)
print(f"After UID join: {len(pub_raw):,} rows, "
      f"{pub_raw.groupby(GROUP_KEYS).ngroups:,} group/BU combinations")

FUNNEL_MAP = {
    'Content Creators & Influencers': 'Upper',
    'Editorial Content':              'Upper',
    'Media Content':                  'Upper',
    'Media Brokers':                  'Upper',
    'Social Traffic':                 'Upper',
    'Social Search':                  'Upper',
    'Retargeting (Display)':          'Upper',
    'Retargeting (Email)':            'Upper',
    'Contextual Targeting':           'Upper',
    'Mobile Traffic':                 'Upper',
    'Mobile Search':                  'Upper',
    'Ad Networks':                    'Upper',
    'Comparison Engine':              'Mid',
    'Comparison Shopping Service (CSS)': 'Mid',
    'Shopping Directory':             'Mid',
    'Lead Generation (Content)':      'Mid',
    'Lead Generation (Email)':        'Mid',
    'Newsletters':                    'Mid',
    'Communities & User-Generated Content': 'Mid',
    'Cashback':                       'Lower',
    'Discount Code':                  'Lower',
    'Loyalty':                        'Lower',
    'Sub Networks':                   'Technical',
    'Direct Linking':                 'Technical',
    'Linking via Landing Pages':      'Technical',
    'Domain Parking':                 'Technical',
    'Direct Traffic':                 'Technical',
    'Virtual Incentives':             'Technical',
}

pub_raw['funnel_stage'] = (
    pub_raw['Publisher Subvertical'].map(FUNNEL_MAP).fillna('Technical')
)

month_cols_2024 = ['Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24',
                   'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24']
month_cols_all  = month_cols_2024 + ['Jan-25']

for col in month_cols_all:
    pub_raw[col] = pd.to_numeric(
        pub_raw[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )
pub_raw[month_cols_all] = pub_raw[month_cols_all].fillna(0)

pub_raw['commission_2024'] = pub_raw[month_cols_2024].sum(axis=1)

funnel_agg = pub_raw.groupby(GROUP_KEYS + ['funnel_stage']).agg(
    commission_2024=('commission_2024', 'sum'),
    commission_jan24=('Jan-24', 'sum'),
    commission_jan25=('Jan-25', 'sum'),
).reset_index()

funnel_pivot = funnel_agg.pivot_table(
    index=GROUP_KEYS,
    columns='funnel_stage',
    values=['commission_2024', 'commission_jan24', 'commission_jan25'],
    fill_value=0
).reset_index()

funnel_pivot.columns = [
    '_'.join(col).strip('_') if col[1] else col[0]
    for col in funnel_pivot.columns
]

comm_cols_2024 = [f'commission_2024_{s}' for s in ['Lower', 'Mid', 'Upper', 'Technical']
                  if f'commission_2024_{s}' in funnel_pivot.columns]
funnel_pivot['pub_total_commission_2024'] = funnel_pivot[comm_cols_2024].sum(axis=1)

for stage in ['Lower', 'Mid', 'Upper', 'Technical']:
    col = f'commission_2024_{stage}'
    if col in funnel_pivot.columns:
        funnel_pivot[f'pub_share_{stage.lower()}_2024'] = (
            funnel_pivot[col] / funnel_pivot['pub_total_commission_2024']
        ).fillna(0)

funnel_pivot['pub_funnel_diversity'] = sum(
    (funnel_pivot[f'commission_2024_{s}'] > 0).astype(int)
    for s in ['Lower', 'Mid', 'Upper', 'Technical']
    if f'commission_2024_{s}' in funnel_pivot.columns
)

jan_cols_24 = [f'commission_jan24_{s}' for s in ['Lower', 'Mid', 'Upper', 'Technical']
               if f'commission_jan24_{s}' in funnel_pivot.columns]
jan_cols_25 = [f'commission_jan25_{s}' for s in ['Lower', 'Mid', 'Upper', 'Technical']
               if f'commission_jan25_{s}' in funnel_pivot.columns]

jan_total_24 = funnel_pivot[jan_cols_24].sum(axis=1).replace(0, np.nan)
jan_total_25 = funnel_pivot[jan_cols_25].sum(axis=1).replace(0, np.nan)

for stage in ['Lower', 'Mid', 'Upper']:
    col24 = f'commission_jan24_{stage}'
    col25 = f'commission_jan25_{stage}'
    if col24 in funnel_pivot.columns and col25 in funnel_pivot.columns:
        share24 = funnel_pivot[col24] / jan_total_24
        share25 = funnel_pivot[col25] / jan_total_25
        funnel_pivot[f'pub_{stage.lower()}_funnel_shift_yoy'] = (
            share25 - share24
        ).fillna(0)

pub_feature_cols = GROUP_KEYS + [
    c for c in funnel_pivot.columns if c.startswith('pub_')
]
publisher_features = funnel_pivot[pub_feature_cols].copy()

print(f"\nPublisher features built: {len(publisher_features):,} groups")
print(f"Features: {[c for c in publisher_features.columns if c.startswith('pub_')]}")

publisher_features.to_csv(OUTPUT_PATH / "publisher_features.csv", index=False)

single_point = panel[panel['ObservationMonth'] == pd.Timestamp('2025-01-01')].copy()
print(f"\nJan 2025 number of rows: {len(single_point):,} rows")

single_point = single_point.merge(publisher_features, on=GROUP_KEYS, how='left')

pub_cols = [c for c in single_point.columns if c.startswith('pub_')]
coverage  = single_point[pub_cols[0]].notna().sum()
print(f"Publisher feature coverage: {coverage:,} of {len(single_point):,} "
      f"({coverage/len(single_point)*100:.1f}%)")

single_point[pub_cols] = single_point[pub_cols].fillna(0)


====PART M - PUBLISHER ENRICHMENT====
Publisher data loaded: 117,648 rows
Columns: ['Advertiser ID', 'Advertiser', 'Advertiser Group', 'Publisher Subvertical', 'Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'Jan-25', 'Unnamed: 17']
After UID join: 116,539 rows, 10,823 group/BU combinations

Publisher features built: 10,823 groups
Features: ['pub_total_commission_2024', 'pub_share_lower_2024', 'pub_share_mid_2024', 'pub_share_upper_2024', 'pub_share_technical_2024', 'pub_funnel_diversity', 'pub_lower_funnel_shift_yoy', 'pub_mid_funnel_shift_yoy', 'pub_upper_funnel_shift_yoy']

Jan 2025 number of rows: 15,247 rows
Publisher feature coverage: 7,170 of 15,247 (47.0%)


In [15]:

print(f"==== FINAL MODELLING DATASET SUMMARY ====")

print(f"Rows:                    {len(single_point):,}")
print(f"Columns:                 {single_point.shape[1]}")
print(f"Decline rate (10%):      {single_point['target_decline'].mean()*100:.1f}%")
print(f"YoY features:            {single_point['has_prior_year_features'].sum():,} of {len(single_point):,}")
print(f"Baseline eligible:       {single_point['baseline_eligible'].sum():,} of {len(single_point):,}")
print(f"Exclusive clients:       {single_point['is_exclusive'].sum():,} of {len(single_point):,}")
print(f"Publisher data coverage: {coverage:,} of {len(single_point):,} "
      f"({coverage/len(single_point)*100:.1f}%)")

print(f"\nDecline rate by exclusive flag:")
print(single_point.groupby('is_exclusive').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean')
).assign(decline_rate=lambda x: x['decline_count']/x['total']*100).round(1).to_string())

print(f"\nDecline rate by publisher data coverage:")
single_point['has_pub_data'] = (single_point['pub_funnel_diversity'] > 0).astype(int)
print(single_point.groupby('has_pub_data').agg(
    total=('target_decline', 'count'),
    decline_count=('target_decline', 'sum'),
    avg_commission=('commission_l12m', 'mean')
).assign(decline_rate=lambda x: x['decline_count']/x['total']*100).round(1).to_string())

single_point.to_csv(OUTPUT_PATH / "modelling_dataset_jan25.csv", index=False)


==== FINAL MODELLING DATASET SUMMARY ====
Rows:                    15,247
Columns:                 81
Decline rate (10%):      40.6%
YoY features:            15,247 of 15,247
Baseline eligible:       15,247 of 15,247
Exclusive clients:       3,599 of 15,247
Publisher data coverage: 7,170 of 15,247 (47.0%)

Decline rate by exclusive flag:
              total  decline_count  avg_commission  decline_rate
is_exclusive                                                    
0             11648         4840.0         26640.1          41.6
1              3599         1351.0        201660.3          37.5

Decline rate by publisher data coverage:
              total  decline_count  avg_commission  decline_rate
has_pub_data                                                    
0              8179         3335.0         12001.8          40.8
1              7068         2856.0        132699.1          40.4


## 15. Follow-up checks on the January 2025 snapshot

Two checks are run on the final snapshot before modelling begins: how many groups the closure filter removed, and what the segment with no publisher data looks like in terms of size, commission concentration and service mix.

In [16]:
print(f"Original Jan 2025 rows (before closure filter): {n_before_closure_jan25:,}")
print(f"After closure filter: {len(single_point):,}")
print(f"Removed: {n_before_closure_jan25 - len(single_point):,}")

has_pub = single_point[single_point['pub_funnel_diversity'] > 0].copy()
no_pub  = single_point[single_point['pub_funnel_diversity'] == 0].copy()

print(f"Has publisher data:  {len(has_pub):,}")
print(f"No publisher data:   {len(no_pub):,}")

print(f"\n=== NO PUBLISHER DATA BREAKDOWN ===")
print(f"\nBy service relationship:")
print(no_pub.groupby('service_relationship_type').agg(
    count=('commission_l12m', 'count'),
    total_commission=('commission_l12m', 'sum'),
    avg_commission=('commission_l12m', 'mean')
).assign(pct_of_no_pub=lambda x: x['count']/len(no_pub)*100).round(1).to_string())

print(f"\nBy commission size tier:")
def tier(x):
    if x < 1_000:     return '1_micro'
    elif x < 10_000:  return '2_small'
    elif x < 100_000: return '3_mid'
    elif x < 1_000_000: return '4_large'
    else:             return '5_strategic'

no_pub['tier'] = no_pub['commission_l12m'].apply(tier)
print(no_pub.groupby('tier').agg(
    count=('commission_l12m', 'count'),
    total_commission=('commission_l12m', 'sum'),
    avg_commission=('commission_l12m', 'mean')
).assign(pct_of_no_pub=lambda x: x['count']/len(no_pub)*100).round(1).to_string())

print(f"\nBy EffectiveBusinessUnit:")
print(no_pub.groupby('EffectiveBusinessUnit').agg(
    count=('commission_l12m', 'count'),
    total_commission=('commission_l12m', 'sum'),
    avg_commission=('commission_l12m', 'mean')
).sort_values('total_commission', ascending=False).round(0).to_string())

total_comm = single_point['commission_l12m'].sum()
no_pub_comm = no_pub['commission_l12m'].sum()
print(f"\nCommission coverage:")
print(f"  Has publisher data: €{has_pub['commission_l12m'].sum():,.0f} "
      f"({has_pub['commission_l12m'].sum()/total_comm*100:.1f}%)")
print(f"  No publisher data:  €{no_pub_comm:,.0f} "
      f"({no_pub_comm/total_comm*100:.1f}%)")

Original Jan 2025 rows (before closure filter): 17,658
After closure filter: 15,247
Removed: 2,411
Has publisher data:  7,068
No publisher data:   8,179

=== NO PUBLISHER DATA BREAKDOWN ===

By service relationship:
                           count  total_commission  avg_commission  pct_of_no_pub
service_relationship_type                                                        
Agency Managed               748        14086796.1         18832.6            9.1
Awin Managed                 355        45014111.3        126800.3            4.3
Self Service                7076        39061676.5          5520.3           86.5

By commission size tier:
             count  total_commission  avg_commission  pct_of_no_pub
tier                                                               
1_micro       4374          947018.1           216.5           53.5
2_small       2465         9062970.4          3676.7           30.1
3_mid         1178        34929716.0         29651.7           14.4
4_large 